# 03 — Solar Geometry Prototype

**Story**: S01-03 | **Spec**: `openspec/capabilities/solar/spec.md`

Computes solar geometry for German Baltic Sea locations using the `suncalc` Python library.
Validates outputs against USNO Solar Calculator reference values.
Produces a sub-hourly daylight/twilight timeline for Go/No-Go planning.

**No credentials required** — all computation is local.

**Reference location**: Kiel, Germany — lat=54.32°N, lon=10.13°E

In [ ]:
import suncalc
from datetime import datetime, date, timezone, timedelta
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.dates as mdates

# Reference location — Warnemünde, Germany
KIEL = dict(lat=54.18, lon=12.08, name="Warnemünde, Germany")

print(f"suncalc version: {suncalc.__version__ if hasattr(suncalc, '__version__') else 'unknown'}")
print(f"Reference location: {KIEL['name']} ({KIEL['lat']}°N, {KIEL['lon']}°E)")

## 1. Solar Events — Sunrise, Sunset, Twilight, Solar Noon

### SCENARIO-SOL-BASIC-001: Warnemünde, 2026-06-21 (summer solstice)

Expected values **in UTC** (Warnemünde = CEST = UTC+2 in summer, 12.08°E):
- Sunrise: ~02:44 UTC (~04:44 CEST) — similar latitude to Kiel
- Sunset:  ~19:57 UTC (~21:57 CEST)
- Solar noon: ~11:12 UTC — 12:00 UTC − (12.08°÷15°/h) = 12:00 − 48 min

> **USNO reference**: https://aa.usno.navy.mil/data/RS_OneYear  
> Set location to Warnemünde (54.18°N, 12.08°E), timezone **UTC**, year 2026.

In [ ]:
def get_solar_events(lat: float, lon: float, d: date) -> dict:
    """Compute all solar events for a given date and location.

    Returns dict with UTC-aware Python datetimes (or None for events that
    don't occur, e.g. polar summer).
    """
    dt = datetime(d.year, d.month, d.day, tzinfo=timezone.utc)
    times = suncalc.get_times(dt, lon, lat)  # NOTE: (lng, lat) order!

    def _utc(v):
        """Convert suncalc value to UTC-aware Python datetime, or None."""
        try:
            if pd.isna(v):       # catches pd.NaT, None, float NaN
                return None
        except (TypeError, ValueError):
            pass
        if isinstance(v, pd.Timestamp):
            # suncalc returns tz-naive Timestamps (UTC implied) — localize then convert
            if v.tzinfo is None:
                v = v.tz_localize("UTC")
            return v.to_pydatetime()   # UTC-aware Python datetime
        if hasattr(v, "tzinfo") and v.tzinfo is not None:
            return v.astimezone(timezone.utc)
        return v

    return {
        "sunrise":           _utc(times.get("sunrise")),
        "sunset":            _utc(times.get("sunset")),
        "solar_noon":        _utc(times.get("solar_noon")),
        "dawn_civil":        _utc(times.get("dawn")),
        "dusk_civil":        _utc(times.get("dusk")),
        "dawn_nautical":     _utc(times.get("nautical_dawn")),
        "dusk_nautical":     _utc(times.get("nautical_dusk")),
        "dawn_astronomical": _utc(times.get("night_end")),
        "dusk_astronomical": _utc(times.get("night")),
        "golden_hour_start": _utc(times.get("golden_hour")),
        "golden_hour_end":   _utc(times.get("golden_hour_end")),
    }

# Test: summer solstice 2026
solstice = date(2026, 6, 21)
events = get_solar_events(KIEL["lat"], KIEL["lon"], solstice)

print(f"Solar events for {KIEL['name']} on {solstice} (UTC):")
for k, v in events.items():
    print(f"  {k:25s}: {v.strftime('%H:%M:%S') + ' UTC' if v is not None else 'None (does not occur)'}")

In [ ]:
# SCENARIO-SOL-BASIC-001: rough sanity check (formal ±1 min validation is in cell-12 vs USNO)
# Warnemünde (12.08°E): solar noon ≈ 12:00 UTC − (12.08°/15°/h) = ~11:12 UTC
# Approximate UTC values for summer solstice (±10 min tolerance — just catching gross errors):
#   Sunrise: ~02:44 UTC  |  Sunset: ~19:50 UTC  |  Solar noon: ~11:12 UTC

def minutes_diff(a: datetime, b: datetime) -> float:
    """Absolute difference in minutes between two UTC datetimes."""
    return abs((a - b).total_seconds()) / 60.0

TOLERANCE_MIN = 10  # ±10 min — rough sanity check only; formal accuracy tested in cell-12

expected_sunrise = datetime(2026, 6, 21,  2, 44, tzinfo=timezone.utc)
expected_sunset  = datetime(2026, 6, 21, 19, 50, tzinfo=timezone.utc)
expected_noon    = datetime(2026, 6, 21, 11, 12, tzinfo=timezone.utc)

sr_diff   = minutes_diff(events["sunrise"],    expected_sunrise)
ss_diff   = minutes_diff(events["sunset"],     expected_sunset)
noon_diff = minutes_diff(events["solar_noon"], expected_noon)

print(f"Sunrise:    computed {events['sunrise'].strftime('%H:%M')} UTC | "
      f"expected ~{expected_sunrise.strftime('%H:%M')} UTC | diff = {sr_diff:.1f} min")
print(f"Sunset:     computed {events['sunset'].strftime('%H:%M')} UTC | "
      f"expected ~{expected_sunset.strftime('%H:%M')} UTC | diff = {ss_diff:.1f} min")
print(f"Solar noon: computed {events['solar_noon'].strftime('%H:%M')} UTC | "
      f"expected ~{expected_noon.strftime('%H:%M')} UTC | diff = {noon_diff:.1f} min")

assert sr_diff <= TOLERANCE_MIN,   f"Sunrise diff {sr_diff:.1f} min — possible TZ or location error"
assert ss_diff <= TOLERANCE_MIN,   f"Sunset diff  {ss_diff:.1f} min — possible TZ or location error"
assert noon_diff <= TOLERANCE_MIN, f"Solar noon diff {noon_diff:.1f} min — possible TZ or location error"
print("\n✓ SCENARIO-SOL-BASIC-001 PASS (sanity check — verify ±1 min against USNO in cell-12)")

## 2. Solar Position — Elevation and Azimuth

### SCENARIO-SOL-ELEVATION-001: Solar elevation at solar noon should be ~58–59°

In [ ]:
def get_solar_position(lat: float, lon: float, dt: datetime) -> dict:
    """Return solar elevation (deg above horizon) and azimuth (deg from north) at a given UTC datetime."""
    pos = suncalc.get_position(dt, lon, lat)  # (lng, lat)
    elevation_deg = np.degrees(pos["altitude"])  # suncalc returns radians
    azimuth_rad   = pos["azimuth"]               # radians, south = 0 in suncalc
    # Convert azimuth to degrees-from-north (standard meteorological convention)
    azimuth_deg = (np.degrees(azimuth_rad) + 180) % 360
    return {"elevation_deg": elevation_deg, "azimuth_deg": azimuth_deg}

# Test at solar noon
pos_noon = get_solar_position(KIEL["lat"], KIEL["lon"], events["solar_noon"])
print(f"Solar position at solar noon ({events['solar_noon'].strftime('%H:%M UTC')}):")
print(f"  Elevation: {pos_noon['elevation_deg']:.2f}°")
print(f"  Azimuth:   {pos_noon['azimuth_deg']:.2f}° from N")

# SCENARIO-SOL-ELEVATION-001: elevation at noon should be 58–59° on solstice at Kiel
assert 57.0 <= pos_noon["elevation_deg"] <= 60.0, \
    f"Solar elevation {pos_noon['elevation_deg']:.2f}° outside expected 57–60° range"
print("\n✓ SCENARIO-SOL-ELEVATION-001 PASS")

## 3. Sub-hourly Solar Timeline — SCENARIO-SOL-TIMELINE-001

15-minute resolution elevation time series over a 48-hour planning window.

In [ ]:
def get_solar_timeline(
    lat: float, lon: float,
    start: datetime, end: datetime,
    resolution_min: int = 15
) -> pd.DataFrame:
    """Return a DataFrame of solar elevation and azimuth at regular intervals."""
    times = pd.date_range(start=start, end=end, freq=f"{resolution_min}min", tz="UTC")
    records = []
    for t in times:
        pos = get_solar_position(lat, lon, t.to_pydatetime())
        records.append({"time": t, **pos})
    return pd.DataFrame(records).set_index("time")

# 48-hour window starting from midnight of the solstice
start_dt = datetime(2026, 6, 21, 0, 0, tzinfo=timezone.utc)
end_dt   = start_dt + timedelta(hours=48)

tl = get_solar_timeline(KIEL["lat"], KIEL["lon"], start_dt, end_dt, resolution_min=15)

print(f"Timeline shape: {tl.shape}")
print(f"First entry: {tl.index[0]}")
print(f"Last entry:  {tl.index[-1]}")
print(f"Elevation range: {tl['elevation_deg'].min():.1f}° to {tl['elevation_deg'].max():.1f}°")

# SCENARIO-SOL-TIMELINE-001: ≥192 data points for 48h at 15-min resolution
assert len(tl) >= 192, f"Expected ≥192 data points, got {len(tl)}"
assert tl["elevation_deg"].between(-90, 90).all(), "Elevation values out of [-90, 90] range"
print(f"\n✓ SCENARIO-SOL-TIMELINE-001 PASS ({len(tl)} data points)")

## 4. Daylight/Twilight/Night Timeline Visualization

In [ ]:
def classify_solar_period(elevation_deg: float) -> str:
    """Classify a solar elevation into a daylight period."""
    if elevation_deg >= 0:
        return "day"
    elif elevation_deg >= -6:
        return "civil_twilight"
    elif elevation_deg >= -12:
        return "nautical_twilight"
    elif elevation_deg >= -18:
        return "astronomical_twilight"
    else:
        return "night"

tl["period"] = tl["elevation_deg"].apply(classify_solar_period)

PERIOD_COLORS = {
    "day":                   "#FFF176",  # yellow
    "civil_twilight":        "#FFB74D",  # amber
    "nautical_twilight":     "#7986CB",  # indigo
    "astronomical_twilight": "#3949AB",  # deep indigo
    "night":                 "#1A237E",  # dark navy
}

fig, (ax_el, ax_bar) = plt.subplots(2, 1, figsize=(14, 7),
                                     gridspec_kw={"height_ratios": [3, 1]}, sharex=True)

# Top panel: elevation curve with filled background by period
for period, color in PERIOD_COLORS.items():
    mask = tl["period"] == period
    if mask.any():
        ax_el.fill_between(tl.index, -20, tl["elevation_deg"],
                           where=mask, color=color, alpha=0.5, step=None)

ax_el.plot(tl.index, tl["elevation_deg"], color="#E65100", lw=1.5, label="Solar elevation")
ax_el.axhline(0, color="gray", lw=1, ls="--")
ax_el.axhline(20, color="green", lw=1, ls=":", label="Min 20° for optical surveys")
ax_el.set_ylabel("Solar elevation (°)")
ax_el.set_ylim(-20, 65)
ax_el.legend(fontsize=9)
ax_el.set_title(f"Solar Geometry — {KIEL['name']} — 48h from {start_dt.strftime('%Y-%m-%d')} UTC")
ax_el.grid(True, alpha=0.3)

# Bottom panel: period colour bar
period_codes = {"day": 4, "civil_twilight": 3, "nautical_twilight": 2,
                "astronomical_twilight": 1, "night": 0}
period_vals = tl["period"].map(period_codes).values
ax_bar.pcolormesh(
    tl.index, [0, 1], [period_vals, period_vals],
    cmap=plt.cm.colors.ListedColormap(list(PERIOD_COLORS.values())[::-1]),
    shading="auto"
)

patches = [mpatches.Patch(color=c, label=p.replace("_", " ").title())
           for p, c in PERIOD_COLORS.items()]
ax_bar.legend(handles=patches, loc="lower center", ncol=5, fontsize=8,
              bbox_to_anchor=(0.5, -0.6))
ax_bar.set_yticks([])
ax_bar.xaxis.set_major_formatter(mdates.DateFormatter("%a %d %b\n%H:%M UTC"))
ax_bar.xaxis.set_major_locator(mdates.HourLocator(byhour=[0, 6, 12, 18]))

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 5. Accuracy Validation — SCENARIO-SOL-ACCURACY-001

Compare suncalc to USNO Solar API for Warnemünde on 2026-09-15.

> **USNO reference values** (manually retrieved from https://aa.usno.navy.mil/data/RS_OneYear):  
> - Set: Lat=54.18, Lon=12.08 E, timezone=**UTC**, year=2026, date=Sep 15  
> - Update `USNO_SUNRISE` and `USNO_SUNSET` below with the values you read off.

*Fill in accurate USNO values below after looking them up.*

In [ ]:
# SCENARIO-SOL-ACCURACY-001: compare to USNO Solar API for 2026-09-15
# Warnemünde (54.18°N, 12.08°E) — CEST (UTC+2) until late Oct
ACCURACY_DATE = date(2026, 9, 15)
events_sep = get_solar_events(KIEL["lat"], KIEL["lon"], ACCURACY_DATE)

print(f"suncalc results for {KIEL['name']} on {ACCURACY_DATE} (UTC):")
print(f"  Sunrise:    {events_sep['sunrise'].strftime('%H:%M:%S')} UTC")
print(f"  Sunset:     {events_sep['sunset'].strftime('%H:%M:%S')} UTC")
print(f"  Solar noon: {events_sep['solar_noon'].strftime('%H:%M:%S')} UTC")
print()
print("USNO Solar API reference (manual lookup):")
print("  → Go to: https://aa.usno.navy.mil/data/RS_OneYear")
print("  → Set: Lat=54.18, Lon=12.08 E, timezone=UTC, year=2026, Sep 15")
print("  → Update USNO_SUNRISE and USNO_SUNSET below")
print()

# UPDATE THESE with actual USNO UTC values after lookup:
USNO_SUNRISE = datetime(2026, 9, 15, 4, 33, tzinfo=timezone.utc)   # placeholder — verify!
USNO_SUNSET  = datetime(2026, 9, 15, 17, 30, tzinfo=timezone.utc)  # placeholder — verify!

sr_acc = minutes_diff(events_sep["sunrise"], USNO_SUNRISE)
ss_acc = minutes_diff(events_sep["sunset"],  USNO_SUNSET)

print(f"Comparison (SCENARIO-SOL-ACCURACY-001, tolerance ±1 min):")
print(f"  Sunrise diff: {sr_acc:.1f} min  {'✓' if sr_acc <= 1.0 else '⚠ update USNO_SUNRISE above with real value'}")
print(f"  Sunset diff:  {ss_acc:.1f} min  {'✓' if ss_acc <= 1.0 else '⚠ update USNO_SUNSET above with real value'}")

## 6. Edge Case — Polar Summer (SCENARIO-SOL-NIGHT-001)

Verify that the solar module handles locations where the sun never sets (no sunset event).
Test location: 70°N (northern Norway — above Arctic Circle in June).

In [ ]:
# SCENARIO-SOL-NIGHT-001: polar summer — sun never sets
POLAR = dict(lat=70.0, lon=25.0, name="Northern Norway (70°N)")

events_polar = get_solar_events(POLAR["lat"], POLAR["lon"], date(2026, 6, 21))

print(f"Solar events for {POLAR['name']} on 2026-06-21:")
for k, v in events_polar.items():
    print(f"  {k:25s}: {v.strftime('%H:%M UTC') if v is not None else 'None (polar day)'}")

# Should handle None gracefully — no crash
sunset_val = events_polar["sunset"]
is_polar_day = sunset_val is None
print(f"\n✓ SCENARIO-SOL-NIGHT-001 PASS: polar_day={is_polar_day} (no crash, sunset={sunset_val})")

## 7. Utility: Solar Elevation at Satellite Overpass Time

This function will be called by the satellite capability (SAT-01) to annotate overpasses.

In [ ]:
def solar_elevation_at_overpass(
    lat: float, lon: float, overpass_time: datetime
) -> float:
    """Return solar elevation (degrees) at a given location and overpass time.
    
    Used to assess whether optical satellite imagery will be well-lit.
    Elevations < 20° may result in poor illumination; < 0° = night pass.
    """
    pos = get_solar_position(lat, lon, overpass_time)
    return pos["elevation_deg"]

# Example: Sentinel-2 overpass over Kiel on 2026-06-21 at 10:05 UTC
test_pass_time = datetime(2026, 6, 21, 10, 5, tzinfo=timezone.utc)
elev = solar_elevation_at_overpass(KIEL["lat"], KIEL["lon"], test_pass_time)
print(f"Solar elevation at Sentinel-2 overpass ({test_pass_time.strftime('%H:%M UTC')}): {elev:.1f}°")
print(f"Illumination quality: {'Excellent (>40°)' if elev > 40 else 'Good (20–40°)' if elev > 20 else 'Poor (<20°)'}")

## 8. Summary of Findings

In [ ]:
print("=== FINDINGS FOR ops/data-sources.md ===")
print()
print("suncalc Python library:")
print("  No network calls — fully local computation")
print("  API: suncalc.get_times(datetime_utc, lng, lat) → dict of pandas Timestamps")
print("       suncalc.get_position(datetime_utc, lng, lat) → {'altitude': rad, 'azimuth': rad}")
print("  NOTE: argument order is (lng, lat) not (lat, lon) — easy gotcha!")
print("  NOTE: altitude and azimuth returned in RADIANS — must convert to degrees")
print("  NOTE: azimuth convention is south=0; convert to north=0 via: (deg(az) + 180) % 360")
print("  NOTE: events that don't occur (e.g., polar summer) are returned as pd.NaT, not None")
print("        → use pd.isna(v) to detect, not 'v is None'")
print()
print("Polar summer edge case:")
print("  get_times() returns pd.NaT for events that don't occur — handled by pd.isna() check")
print("  Our get_solar_events() wrapper converts NaT → None correctly")
print()
print("Phase 2 design notes:")
print("  - get_solar_events() and get_solar_timeline() ready for src/boatops/services/solar.py")
print("  - solar_elevation_at_overpass() reused directly by satellite service")
print("  - No caching needed (fast local computation)")
print("  - TODO: verify USNO accuracy by running SCENARIO-SOL-ACCURACY-001 with real USNO lookup")